# NB8 — Agents as consumers, and illustrative data provenance

**Stack:** `deltalake` + DuckDB. No LLM call, no API key — the agent loop is
deterministic so the *data contract* is what you study, not model behaviour.
Maps to slide §11 (Agent Memory & Trajectory, MCP 2026-07-28) + §12
(Provenance) + deliverable bullet 8.

Three parts:

1. **Trajectories** — the lakehouse as system-of-record for what an agent did
2. **MCP-inspired simulation** — caching, confirmation and task polling
3. **Provenance** — record origins, classify rows and pin a corpus version

This notebook does not implement a conforming MCP server or certify legal
compliance. Its four buckets and simplified license mapping are lab rules.

In [1]:
import _setup  # noqa: F401

import json
import time
from pathlib import Path

import duckdb
import polars as pl
import pyarrow as pa
from deltalake import DeltaTable, write_deltalake

from lakehouse import catalog, namespace, path, reset, reset_catalog, to_arrow

import generate_ai_data as gen

TRACES = path("bronze", "agent_traces")
DOCS = path("bronze", "docs_multimodal")
if not Path(TRACES).exists() or not Path(DOCS).exists():
    gen.main()

con = duckdb.connect()
con.register("bronze_traces", DeltaTable(TRACES).to_pyarrow_table())
print(f"Bronze trajectory steps: {DeltaTable(TRACES).count():,}")

Bronze trajectory steps: 1,578


### 📝 Nhận xét — Bronze trajectory

- **1.578 bước** (step) của agent ở tầng Bronze, mỗi dòng là một bước trong một session: tool gọi, trạng thái, reward, token, latency, `subject_id`.
- Agent loop ở đây **tất định, không gọi LLM** — mục tiêu là nghiên cứu **hợp đồng dữ liệu** (lưu gì, version thế nào, ai được làm gì), không phải hành vi model.


## Part 1 — Trajectories through the medallion

> A **trajectory** (rollout) is a sequence of `(observation, action, reward)`
> from the initial state to termination — the fuel an RL update consumes.

It differs from supervised data in one way that changes your storage design:
**the data distribution shifts as the policy improves**, so a static dataset
is useless. Trajectory tables are therefore *append-heavy, versioned, and
sliceable by policy version.*

Bronze = raw steps · Silver = parsed + `trajectory_id` · Gold = per-agent rollups.

In [2]:
SILVER = path("silver", "agent_trajectories")
GOLD = path("gold", "agent_performance")
reset(SILVER, GOLD)

# Two policy versions in flight — the partition key the slide insists on.
silver = to_arrow(con.sql("""
    SELECT
        session_id                        AS trajectory_id,
        step,
        tool,
        status,
        reward,
        subject_id,
        input_tokens + output_tokens      AS total_tokens,
        latency_ms,
        (input_tokens * 3.0 + output_tokens * 15.0) / 1e6 AS cost_usd,
        CASE WHEN CAST(substr(session_id, 6) AS INT) < 150
             THEN 'policy-v2' ELSE 'policy-v3' END        AS agent_version
    FROM bronze_traces
"""))

# Partitioning by agent_version lets you drop or retrain on one policy's
# rollouts without touching the other's.
write_deltalake(SILVER, silver, mode="overwrite", partition_by=["agent_version"])
con.register("silver", DeltaTable(SILVER).to_pyarrow_table())
print(f"Silver: {silver.num_rows:,} steps, partitioned by agent_version")
print(f"  partitions on disk: {sorted(p.name for p in Path(SILVER).glob('agent_version=*'))}")

Silver: 1,578 steps, partitioned by agent_version
  partitions on disk: ['agent_version=policy-v2', 'agent_version=policy-v3']


### 📝 Nhận xét — Silver partition theo `agent_version`

- Silver đổi `session_id` → `trajectory_id`, tính `total_tokens`, `cost_usd` (giá minh hoạ $3/$15 mỗi 1M token) và gán **`agent_version`**: session < 150 → `policy-v2`, còn lại → `policy-v3` (quy tắc gán của lab, mô phỏng hai policy cùng chạy).
- Ghi **partition theo `agent_version`** → 2 thư mục `agent_version=policy-v2/` và `=policy-v3/` ✅.
- Vì sao partition theo policy: dữ liệu RL **thay đổi phân phối khi policy cải thiện** — rollout của policy cũ không còn đại diện. Partition cho phép **loại bỏ, huấn luyện lại hoặc so sánh** từng policy bằng thao tác trên partition, không đụng tới policy kia. Bảng trajectory vì thế là *append-heavy, có version, và cắt được theo policy*.


In [3]:
gold = to_arrow(con.sql("""
    WITH per_traj AS (
        SELECT trajectory_id, agent_version,
               max(reward)              AS success,
               count(*)                 AS steps,
               sum(cost_usd)            AS cost_usd,
               sum(latency_ms)          AS latency_ms,
               max(status = 'error')    AS had_error
        FROM silver GROUP BY 1, 2
    )
    SELECT agent_version,
           count(*)                              AS trajectories,
           round(avg(success), 3)                AS success_rate,
           round(avg(steps), 2)                  AS avg_steps,
           round(avg(cost_usd), 5)               AS avg_cost_usd,
           round(sum(cost_usd), 2)               AS total_cost_usd,
           round(avg(latency_ms) / 1000, 1)      AS avg_seconds
    FROM per_traj GROUP BY 1 ORDER BY 1
"""))
write_deltalake(GOLD, gold, mode="overwrite")
print(pl.from_arrow(gold))

shape: (2, 7)
┌──────────────┬──────────────┬──────────────┬───────────┬─────────────┬─────────────┬─────────────┐
│ agent_versio ┆ trajectories ┆ success_rate ┆ avg_steps ┆ avg_cost_us ┆ total_cost_ ┆ avg_seconds │
│ n            ┆ ---          ┆ ---          ┆ ---       ┆ d           ┆ usd         ┆ ---         │
│ ---          ┆ i64          ┆ f64          ┆ f64       ┆ ---         ┆ ---         ┆ f64         │
│ str          ┆              ┆              ┆           ┆ f64         ┆ f64         ┆             │
╞══════════════╪══════════════╪══════════════╪═══════════╪═════════════╪═════════════╪═════════════╡
│ policy-v2    ┆ 150          ┆ 0.76         ┆ 5.26      ┆ 0.06915     ┆ 10.37       ┆ 16.2        │
│ policy-v3    ┆ 150          ┆ 0.753        ┆ 5.26      ┆ 0.06928     ┆ 10.39       ┆ 15.7        │
└──────────────┴──────────────┴──────────────┴───────────┴─────────────┴─────────────┴─────────────┘


### 📝 Nhận xét — Gold: hiệu năng theo policy

- Gold tổng hợp hai bước: theo **trajectory** (success = `max(reward)`, số bước, chi phí, latency, có lỗi không) rồi theo **policy**. Có **2 dòng, đủ cả hai policy** ✅.
- Mỗi policy 150 trajectory; success rate **0,760 (v2)** và **0,753 (v3)**, ~5,26 bước, ~$0,069 và ~16 giây mỗi trajectory, tổng ~$10,4 mỗi policy.
- Hai policy gần như **giống hệt nhau** — vì dữ liệu tổng hợp và `agent_version` được gán theo số session, không phải hai model khác nhau thật. Gold cho phép so sánh, nhưng **không được kết luận "v2 tốt hơn"** từ chênh lệch 0,007 trên 150 mẫu (nằm trong nhiễu thống kê).


### The reproducibility contract: pin the table version into the run

The slide's rule: *pin the trajectory table version into the training run* —
the same contract as MLflow ↔ Delta version. Without it, "which data did this
policy train on?" has no answer, and Annex IV has no answer either.

In [4]:
training_run = {
    "run_id": "rl-run-2026-08-17-001",
    "policy": "policy-v4",
    "trajectory_table": SILVER,
    "table_version": DeltaTable(SILVER).version(),     # ← the pin
    "n_steps_seen": DeltaTable(SILVER).count(),
}
print(json.dumps(training_run, indent=2, default=str))

# The world moves on: new rollouts land.
write_deltalake(SILVER, silver.slice(0, 400), mode="append", partition_by=["agent_version"])
print(f"\nAfter more rollouts land — table version {DeltaTable(SILVER).version()}, "
      f"{DeltaTable(SILVER).count():,} steps")

# Six months later, an auditor asks what the run actually saw.
pinned = DeltaTable(SILVER, version=training_run["table_version"])
print(f"Replay at pinned version {training_run['table_version']}: {pinned.count():,} steps")
print(f"Matches what training saw: {pinned.count() == training_run['n_steps_seen']}")
print("\nThat one integer is the difference between a reproducible run and a story.")

{
  "run_id": "rl-run-2026-08-17-001",
  "policy": "policy-v4",
  "trajectory_table": "C:\\Users\\Admin\\Desktop\\ai-app-lab\\L18-D18-work\\K4-Track02-Day18-LeDucTung-03005-Lakehouse-Lab\\_lakehouse\\silver\\agent_trajectories",
  "table_version": 0,
  "n_steps_seen": 1578
}

After more rollouts land — table version 1, 1,978 steps
Replay at pinned version 0: 1,578 steps
Matches what training saw: True

That one integer is the difference between a reproducible run and a story.


### 📝 Nhận xét — Pin version vào training run

- Training run ghi `table_version = 0` và `n_steps_seen = 1.578` — một số nguyên là đủ để định danh **chính xác** dữ liệu đã dùng (giống hợp đồng MLflow ↔ Delta version).
- Sau đó bảng tiếp tục nhận dữ liệu: **version 1, 1.978 bước**. Sáu tháng sau, kiểm toán viên replay ở **version 0 → 1.578 bước = khớp** ✅.
- Không có pin, câu hỏi "policy này được train trên dữ liệu nào?" không có câu trả lời — bảng đã thay đổi. Đây là nền tảng của reproducibility và của tài liệu kỹ thuật cho kiểm toán.


In [5]:
# Replay gốc chỉ so SỐ DÒNG. Kiểm tra thêm NỘI DUNG bằng fingerprint thứ tự-độc-lập
KEY = ["trajectory_id", "step"]

def fingerprint(tbl) -> int:
    """Tổng hash của từng dòng (sắp xếp theo khoá) — đổi 1 giá trị là đổi fingerprint."""
    df = pl.from_arrow(tbl).sort(KEY)
    return int(df.hash_rows(seed=0).sum())

fp_training = fingerprint(silver)                                   # dữ liệu lúc train
fp_pinned   = fingerprint(DeltaTable(SILVER, version=training_run["table_version"]).to_pyarrow_table())
current     = pl.from_arrow(DeltaTable(SILVER).to_pyarrow_table())
print(f"fingerprint lúc train : {fp_training}")
print(f"fingerprint replay v{training_run['table_version']}: {fp_pinned}   → nội dung khớp: {fp_training == fp_pinned}")
print(f"fingerprint hiện tại  : {fingerprint(DeltaTable(SILVER).to_pyarrow_table())}")
dups = current.height - current.select(KEY).n_unique()
print(f"\nBảng hiện tại: {current.height:,} dòng nhưng chỉ {current.select(KEY).n_unique():,} cặp (trajectory_id, step) duy nhất"
      f" → {dups:,} dòng 'rollout mới' là bản sao")

fingerprint lúc train : 18358778902051253894
fingerprint replay v0: 18358778902051253894   → nội dung khớp: True
fingerprint hiện tại  : 16457572585367730947

Bảng hiện tại: 1,978 dòng nhưng chỉ 1,578 cặp (trajectory_id, step) duy nhất → 400 dòng 'rollout mới' là bản sao


### 📝 Nhận xét — Replay theo nội dung, không chỉ số dòng

- Check gốc của notebook chỉ so **số dòng** (giới hạn được ghi rõ trong CHECKPOINTS). Cell trên bổ sung **fingerprint nội dung**: tổng hash của từng dòng sau khi sắp theo `(trajectory_id, step)`.
- Fingerprint lúc train **=** fingerprint replay ở v0 ⇒ nội dung **khớp hoàn toàn**, không chỉ số lượng. Fingerprint version hiện tại khác ⇒ phát hiện được bảng đã đổi.
- Phát hiện: 400 dòng "rollout mới" là **bản sao** của 400 dòng đầu (`silver.slice(0, 400)`) — bảng hiện tại có 1.978 dòng nhưng chỉ 1.578 cặp khoá duy nhất. Một check chỉ đếm dòng sẽ coi đây là dữ liệu mới hợp lệ. Bài học: training run nên ghi **cả version lẫn fingerprint nội dung**, và append trajectory nên idempotent (MERGE theo khoá, NB3).


## Part 2 — MCP: the shape of the agent ↔ lakehouse boundary

**MCP revision 2026-07-28** made changes that matter specifically to a data
platform. The table summarizes ideas from the official release; the class
below is a simplified local simulation, not the protocol's wire schema.
Source: [MCP release](https://blog.modelcontextprotocol.io/posts/2026-07-28/).

| Change | Why a data team cares |
|---|---|
| **Stateless core** — no `initialize` handshake; each request self-describes in `_meta` | A catalog MCP server can sit behind a round-robin load balancer with no session store |
| **Cacheable lists** — list responses advertise cache hints | A catalog need not re-list itself every agent turn |
| **Multi-round-trip** — `resultType: input_required` | Human-in-the-loop before `DELETE` or a cross-border export |
| **Header routing** — `Mcp-Method`, `Mcp-Name` | Gateway routes and **meters per tool** without parsing JSON |
| **Tasks extension** — poll `tasks/get` | The right shape for a 40-minute Spark job |

In [6]:
CAT = "nb8"          # own catalog dir — see scripts/lakehouse.py:_catalog_dir
reset_catalog(CAT)
cat = catalog(CAT)
ns = namespace(cat, "lake")
ice = cat.create_table(f"{ns}.trajectories", schema=pa.schema([
    pa.field("trajectory_id", pa.string(), nullable=False),
    pa.field("agent_version", pa.string()),
    pa.field("reward", pa.float64()),
]))
ice.append(pa.table({
    "trajectory_id": to_arrow(con.sql("SELECT DISTINCT trajectory_id FROM silver ORDER BY 1")).column(0),
    "agent_version": to_arrow(con.sql("SELECT DISTINCT ON (trajectory_id) agent_version FROM silver ORDER BY trajectory_id")).column(0),
    "reward": to_arrow(con.sql("SELECT max(reward) FROM silver GROUP BY trajectory_id ORDER BY trajectory_id")).column(0),
}, schema=ice.schema().as_arrow()))
print(f"Catalog now serves: {cat.list_tables(ns)}")

Catalog now serves: [('lake', 'trajectories')]


### 📝 Nhận xét — Catalog phục vụ agent

- Bảng Iceberg `lake.trajectories` (300 trajectory: id, policy, reward) được đăng ký **qua catalog** — agent khám phá dữ liệu bằng cách hỏi catalog, không biết đường dẫn file (như NB5).
- Phần 2 mô phỏng **một số ý tưởng** của MCP revision 2026-07-28 ở dạng lớp Python cục bộ — **không phải MCP server**, không có transport hay wire schema thật.


In [7]:
class LakehouseMCP:
    """An MCP-2026-07-28-shaped read surface over the catalog.

    Deliberately NOT a network server — the point is the contract, not the
    transport. Field names and responses are simplified for this lab.
    The cache is in list_tables; confirmed is caller-controlled, not an
    authorization boundary; delete_rows is a no-op.
    """

    DESTRUCTIVE = {"drop_table", "delete_rows"}

    def __init__(self, cat, ns: str, list_ttl_ms: int = 60_000):
        self.cat, self.ns = cat, ns
        self.list_ttl_ms = list_ttl_ms
        self._cache: dict[str, tuple[float, object]] = {}
        self.meter: dict[str, dict] = {}          # per-tool billing, keyed by Mcp-Name
        self.catalog_reads = 0                    # how often we really hit the catalog

    # ── Cacheable lists: tools/list advertises its own TTL and scope ──
    def tools_list(self) -> dict:
        return {
            "tools": [
                {"name": "list_tables",  "description": "Enumerate tables in a namespace"},
                {"name": "get_schema",   "description": "Column names and types"},
                {"name": "query",        "description": "Read-only SQL over a table"},
                {"name": "submit_scan",  "description": "Long scan; returns a task handle"},
                {"name": "delete_rows",  "description": "DESTRUCTIVE: delete matching rows"},
            ],
            # Lab-specific metadata, not a complete MCP cache-hints schema.
            "_meta": {"ttlMs": self.list_ttl_ms, "cacheScope": "session"},
        }

    # ── Stateless core: no handshake, no Mcp-Session-Id; _meta per request ──
    def call(self, name: str, args: dict | None = None, _meta: dict | None = None) -> dict:
        args, _meta = args or {}, _meta or {}
        t0 = time.perf_counter()

        # Header-style routing + metering. A gateway can do this without
        # parsing the JSON body at all.
        headers = {"Mcp-Method": "tools/call", "Mcp-Name": name}

        # Multi-round-trip: destructive tools stop and ask a human first.
        if name in self.DESTRUCTIVE and not _meta.get("confirmed"):
            return {"resultType": "input_required",
                    "prompt": f"Confirm {name}({args}) — this cannot be undone by the agent.",
                    "_meta": {"headers": headers, "requiresConfirmation": True}}

        result = self._dispatch(name, args)
        elapsed_ms = (time.perf_counter() - t0) * 1000
        m = self.meter.setdefault(name, {"calls": 0, "ms": 0.0})
        m["calls"] += 1
        m["ms"] += elapsed_ms
        return {"resultType": "ok", "result": result,
                "_meta": {"headers": headers, "elapsedMs": round(elapsed_ms, 2)}}

    def _dispatch(self, name: str, args: dict):
        if name == "list_tables":
            key = f"list:{self.ns}"
            hit = self._cache.get(key)
            if hit and (time.time() - hit[0]) * 1000 < self.list_ttl_ms:
                return {"tables": hit[1], "cached": True}
            self.catalog_reads += 1
            tables = [f"{a}.{b}" for a, b in self.cat.list_tables(self.ns)]
            self._cache[key] = (time.time(), tables)
            return {"tables": tables, "cached": False}

        if name == "get_schema":
            t = self.cat.load_table(args["table"])
            return {"columns": [{"name": f.name, "type": str(f.field_type),
                                 "required": f.required} for f in t.schema().fields]}

        if name == "query":
            t = self.cat.load_table(args["table"])
            # Guardrail: the agent never gets an unbounded scan.
            limit = min(int(args.get("limit", 10)), 100)
            return {"rows": t.scan(limit=limit).to_arrow().to_pylist()}

        if name == "submit_scan":
            # Tasks extension: return a handle immediately, poll for completion.
            # This lab emulates delayed completion; it does not run a background job.
            task_id = f"task_{len(self._cache):04d}"
            self._cache[task_id] = (time.time(), args["table"])
            return {"taskId": task_id, "status": "working", "pollAfterMs": 50}

        if name == "delete_rows":
            return {"deleted": 0, "note": "wired to a no-op in the lab"}

        raise ValueError(f"unknown tool: {name}")

    def tasks_get(self, task_id: str) -> dict:
        created, table = self._cache[task_id]
        if (time.time() - created) * 1000 < 50:
            return {"status": "working"}
        n = self.cat.load_table(table).scan().to_arrow().num_rows
        return {"status": "completed", "result": {"rows": n}}


mcp = LakehouseMCP(cat, ns)

### 📝 Nhận xét — Lớp `LakehouseMCP` mô phỏng

- **Stateless:** mỗi `call()` tự mang `_meta`, không có handshake/session ⇒ có thể đặt sau load balancer.
- **Header routing:** `Mcp-Method` / `Mcp-Name` ⇒ gateway định tuyến và **tính tiền theo tool** mà không cần parse body.
- **Guardrail:** `query` bị giới hạn `limit ≤ 100` — agent không bao giờ được scan không giới hạn.
- **Giới hạn đã khai báo:** cache nằm ở `list_tables`, cờ `confirmed` do bên gọi tự truyền, `delete_rows` là no-op.


### Cached list_tables calls: avoid repeated catalog reads

In [8]:
for turn in range(5):
    r = mcp.call("list_tables")
    print(f"  turn {turn}: cached={r['result']['cached']}  tables={r['result']['tables']}")
print(f"\nActual catalog round-trips for 5 agent turns: {mcp.catalog_reads}")
print(f"tools/list advertises: {mcp.tools_list()['_meta']}")

  turn 0: cached=False  tables=['lake.trajectories']
  turn 1: cached=True  tables=['lake.trajectories']
  turn 2: cached=True  tables=['lake.trajectories']
  turn 3: cached=True  tables=['lake.trajectories']
  turn 4: cached=True  tables=['lake.trajectories']

Actual catalog round-trips for 5 agent turns: 1
tools/list advertises: {'ttlMs': 60000, 'cacheScope': 'session'}


### 📝 Nhận xét — Cache `list_tables`: 5 lượt → 1 lần đọc catalog

- Lượt 0 `cached=False` (đọc catalog), lượt 1–4 `cached=True` ⇒ **5 lượt agent chỉ 1 round-trip tới catalog** ✅. Agent hỏi lại danh sách bảng ở mỗi lượt là hành vi rất phổ biến; cache với TTL (60 s, scope session) tránh dội tải vào catalog.
- Lưu ý: cache được đo ở **`list_tables`**, không phải `tools/list` (kết quả `tools/list` chỉ *quảng bá* `ttlMs`).
- Đánh đổi: trong 60 s, bảng mới tạo sẽ **không hiện ra** với agent (dữ liệu cũ trong cache) — TTL phải cân bằng giữa tải catalog và độ tươi.


### Human-in-the-loop before anything destructive

In [9]:
attempt = mcp.call("delete_rows", {"table": f"{ns}.trajectories", "where": "reward = 0"})
print(f"resultType: {attempt['resultType']}")
print(f"prompt:     {attempt['prompt']}")

approved = mcp.call("delete_rows", {"table": f"{ns}.trajectories", "where": "reward = 0"},
                    _meta={"confirmed": True})
print(f"\nafter human approval → resultType: {approved['resultType']}")
print("\nLab limitation: confirmed is supplied by the caller; this is not an authorization boundary.")

resultType: input_required
prompt:     Confirm delete_rows({'table': 'lake.trajectories', 'where': 'reward = 0'}) — this cannot be undone by the agent.

after human approval → resultType: ok

Lab limitation: confirmed is supplied by the caller; this is not an authorization boundary.


### 📝 Nhận xét — Human-in-the-loop trước thao tác phá huỷ

- Lần gọi đầu `delete_rows` **không có xác nhận** ⇒ `resultType: input_required` kèm prompt yêu cầu xác nhận ✅ — tool không chạy.
- Gọi lại với `_meta={"confirmed": True}` ⇒ `resultType: ok` (nhưng `delete_rows` là no-op, không xoá gì).
- ⚠️ **Giới hạn quan trọng:** `confirmed` do **chính bên gọi** truyền vào ⇒ agent có thể **tự xác nhận** cho chính mình. Đây **không phải ranh giới phân quyền**. Hệ thống thật cần xác nhận **ngoài băng** (out-of-band): người duyệt qua UI riêng, token ký bởi người duyệt, hoặc quyền IAM tách biệt — agent không thể tự tạo ra.


### Tasks: the right shape for a scan that takes 40 minutes

In [10]:
sub = mcp.call("submit_scan", {"table": f"{ns}.trajectories"})
task_id = sub["result"]["taskId"]
print(f"submit_scan → {sub['result']}")
for poll in range(5):
    st = mcp.tasks_get(task_id)
    print(f"  tasks/get #{poll}: {st['status']}")
    if st["status"] == "completed":
        print(f"  result: {st['result']}")
        break
    time.sleep(0.03)
print("\nThe task handle and polling are simulated locally; no background Spark job is submitted.")

submit_scan → {'taskId': 'task_0001', 'status': 'working', 'pollAfterMs': 50}
  tasks/get #0: working
  tasks/get #1: working
  tasks/get #2: completed
  result: {'rows': 300}

The task handle and polling are simulated locally; no background Spark job is submitted.


### 📝 Nhận xét — Tasks: submit → poll → completed

- `submit_scan` trả **ngay** một task handle (`status: working`, `pollAfterMs: 50`); client poll `tasks_get` vài lần cho tới `completed` với kết quả **300 dòng** ✅.
- Đây là hình dạng đúng cho job dài (Spark 40 phút): agent không giữ kết nối mở, không timeout, có thể làm việc khác rồi quay lại.
- Giới hạn: hoàn thành được **giả lập bằng độ trễ 50 ms**, không có job nền thật.


### Per-tool metering — the FinOps hook

Because `Mcp-Name` is a header, a gateway can bill per tool without ever
parsing the request body.

In [11]:
mcp.call("get_schema", {"table": f"{ns}.trajectories"})
mcp.call("query", {"table": f"{ns}.trajectories", "limit": 3})
print(f"{'tool':<14} {'calls':>6} {'total ms':>10}")
for tool, m in sorted(mcp.meter.items()):
    print(f"{tool:<14} {m['calls']:>6} {m['ms']:>10.2f}")

tool            calls   total ms
delete_rows         1       0.01
get_schema          1       2.59
list_tables         5       2.73
query               1      10.77
submit_scan         1       0.01


### 📝 Nhận xét — Metering theo tool (FinOps)

- `meter` đếm số lần gọi và tổng thời gian **theo tên tool** (khoá chính là `Mcp-Name`): `list_tables` 5 lần, `get_schema`/`query`/`submit_scan`/`delete_rows` mỗi tool 1 lần. Lần `delete_rows` bị chặn để chờ xác nhận không được tính — chỉ lần đã xác nhận.
- Với tên tool nằm trong header, gateway có thể **tính chi phí từng tool cho từng agent/tenant** mà không cần parse JSON — cơ sở cho chargeback và phát hiện agent lạm dụng tool đắt.


## Part 3 — Provenance as recorded data, not a compliance certificate

[EU AI Act, Article 10](https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:02024R1689-20260727)
addresses data governance and quality for relevant high-risk AI systems.
It does not prescribe the four buckets used in this lab. Applicability and
deadlines require checking the current law and the particular system.

Here we illustrate a governed column plus a partition key. The existing
mapping below is intentionally limited: CC BY 4.0 is a license requiring
attribution, not public domain; user ownership plus consent does not prove
a scraping opt-out check. Source: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).
Treat these labels as lab fixtures, not evidence of real training rights.

In [12]:
docs = DeltaTable(DOCS).to_pyarrow_table()
con.register("docs", docs)

# The four illustrative lab buckets, as ONE column expression. Anything that falls
# through is UNCLASSIFIED — and UNCLASSIFIED is an audit finding, not a
# rounding error, so it must never silently become a default bucket.
BUCKET_SQL = """
    CASE
        WHEN license IN ('proprietary', 'commercial')          THEN 'licensed'
        WHEN license = 'cc-by-4.0'                             THEN 'public_domain'
        WHEN license = 'user-owned'  AND consent_train         THEN 'scraped_optout_checked'
        WHEN license = 'synthetic'   AND generator IS NOT NULL THEN 'synthetic'
        ELSE 'UNCLASSIFIED'
    END
"""
audit = to_arrow(con.sql(f"""
    SELECT {BUCKET_SQL} AS provenance_bucket,
           count(*) AS rows,
           round(100.0 * count(*) / sum(count(*)) OVER (), 1) AS pct
    FROM docs GROUP BY 1 ORDER BY rows DESC
"""))
print(pl.from_arrow(audit))

unclassified = con.sql(f"SELECT count(*) FROM docs WHERE {BUCKET_SQL} = 'UNCLASSIFIED'").fetchone()[0]
print(f"\nUNCLASSIFIED rows: {unclassified:,}")
print("→ UNCLASSIFIED rows fail the lab's classification rule; this is not a legal audit result.")

shape: (5, 3)
┌────────────────────────┬──────┬──────┐
│ provenance_bucket      ┆ rows ┆ pct  │
│ ---                    ┆ ---  ┆ ---  │
│ str                    ┆ i64  ┆ f64  │
╞════════════════════════╪══════╪══════╡
│ licensed               ┆ 675  ┆ 33.8 │
│ UNCLASSIFIED           ┆ 334  ┆ 16.7 │
│ public_domain          ┆ 333  ┆ 16.7 │
│ synthetic              ┆ 331  ┆ 16.6 │
│ scraped_optout_checked ┆ 327  ┆ 16.4 │
└────────────────────────┴──────┴──────┘

UNCLASSIFIED rows: 334
→ UNCLASSIFIED rows fail the lab's classification rule; this is not a legal audit result.


### 📝 Nhận xét — Phân loại provenance (minh hoạ)

- Một biểu thức `CASE` duy nhất gán mỗi tài liệu vào 4 bucket của lab: **licensed 675 (33,8%)**, **public_domain 333**, **synthetic 331**, **scraped_optout_checked 327**; phần còn lại **UNCLASSIFIED 334 (16,7%)** — đúng 334 tài liệu `license = unknown` (cũng là nhóm bị loại trong filtered search ở NB7).
- **UNCLASSIFIED là phát hiện kiểm toán, không phải sai số làm tròn**: không bao giờ được âm thầm gán vào bucket mặc định.
- ⚠️ **Giới hạn của mapping (lab rule, không phải pháp lý):**
  - `cc-by-4.0` → `public_domain` là **sai về bản chất**: CC BY 4.0 là giấy phép **có điều kiện ghi công**, không phải public domain.
  - `user-owned` + `consent_train` → `scraped_optout_checked` **không chứng minh** đã kiểm tra opt-out khi scraping.
  - EU AI Act Điều 10 yêu cầu quản trị dữ liệu cho hệ thống rủi ro cao nhưng **không quy định 4 bucket này**. Không dùng mapping này để kết luận quyền sử dụng dữ liệu thật.


### Make the bucket a real column *and* a partition key

A provenance partition key can support file pruning when the reader uses
Delta's partition metadata. The DuckDB queries here read a registered Arrow
table, so this cell does not measure partition pruning.

In [13]:
GOVERNED = path("silver", "training_corpus_governed")
reset(GOVERNED)
governed = to_arrow(con.sql(f"""
    SELECT doc_id, title, topic, subject_id, source, license, consent_train, generator,
           {BUCKET_SQL} AS provenance_bucket
    FROM docs
"""))
write_deltalake(GOVERNED, governed, mode="overwrite", partition_by=["provenance_bucket"])

parts = sorted(p.name for p in Path(GOVERNED).glob("provenance_bucket=*"))
print("Partitions on disk:")
for p_ in parts:
    print(f"  {p_}")

con.register("governed", DeltaTable(GOVERNED).to_pyarrow_table())
trainable = con.sql("""SELECT count(*) FROM governed
                       WHERE provenance_bucket <> 'UNCLASSIFIED'""").fetchone()[0]
print(f"\nRows selected by the lab's training filter: {trainable:,} / {governed.num_rows:,}")
print(f"Excluded as UNCLASSIFIED:  {governed.num_rows - trainable:,}")

Partitions on disk:
  provenance_bucket=UNCLASSIFIED
  provenance_bucket=licensed
  provenance_bucket=public_domain
  provenance_bucket=scraped_optout_checked
  provenance_bucket=synthetic



Rows selected by the lab's training filter: 1,666 / 2,000
Excluded as UNCLASSIFIED:  334


### 📝 Nhận xét — Bucket là cột và là partition key

- `provenance_bucket` thành **cột thật** và **partition key**: 5 thư mục trên đĩa — 4 bucket của lab + `UNCLASSIFIED` ✅.
- Tập trainable = mọi bucket trừ `UNCLASSIFIED` ⇒ **1.666 / 2.000** dòng; **334 bị loại** ✅.
- Lợi ích của partition: reader dùng partition metadata của Delta có thể bỏ qua cả thư mục `UNCLASSIFIED`, và xoá/khoá một nguồn dữ liệu là thao tác trên partition. (DuckDB ở đây đọc qua Arrow nên cell này **không đo** pruning.)
- Provenance phải được **ghi từ lúc ingest** — không thể suy ngược nguồn gốc sau khi dữ liệu đã trộn vào corpus.


### A version record: "which corpus version did this simulated run use?"

`DESCRIBE HISTORY` + the pinned version + the run id. Three facts, one query.

In [14]:
corpus_version = DeltaTable(GOVERNED).version()
model_card = {
    "model": "vinuni-rag-v1",
    "corpus_table": "silver.training_corpus_governed",
    "corpus_version": corpus_version,
    "rows_used": trainable,
    "buckets_used": [p_.split("=", 1)[1] for p_ in parts if "UNCLASSIFIED" not in p_],
    "excluded_rows": governed.num_rows - trainable,
    "exclusion_reason": "license=unknown → excluded by the lab classification rule",
}
print(json.dumps(model_card, indent=2))

hist = DeltaTable(GOVERNED).history()
print(f"\nDESCRIBE HISTORY → {len(hist)} version(s); v{corpus_version} written "
      f"by {hist[0]['operation']}")

{
  "model": "vinuni-rag-v1",
  "corpus_table": "silver.training_corpus_governed",
  "corpus_version": 0,
  "rows_used": 1666,
  "buckets_used": [
    "licensed",
    "public_domain",
    "scraped_optout_checked",
    "synthetic"
  ],
  "excluded_rows": 334,
  "exclusion_reason": "license=unknown \u2192 excluded by the lab classification rule"
}

DESCRIBE HISTORY → 1 version(s); v0 written by WRITE


### 📝 Nhận xét — Model card ghi version corpus

- Model card nối ba sự thật: **tên bảng corpus**, **`corpus_version = 0`**, và số dòng/bucket đã dùng (1.666 dòng, 4 bucket, 334 bị loại với lý do). `DESCRIBE HISTORY` xác nhận v0 do `WRITE` tạo.
- Nhờ đó "model này học từ dữ liệu nào, dữ liệu đó có nguồn gốc gì?" trả lời được bằng một truy vấn — điều kiện cần cho tài liệu kỹ thuật và kiểm toán.


### Subject deletion: distinguish the current table from older versions

We simulate a subject-deletion request and locate the affected rows.
The test only proves deletion from the current table version. It does not
cover backups, external indexes, trained models or legal obligations.

In [15]:
SUBJECT = "user_007"
before = con.sql(f"SELECT count(*) FROM governed WHERE subject_id = '{SUBJECT}'").fetchone()[0]

# What was this subject's data used for? Provenance answers it directly.
usage = to_arrow(con.sql(f"""
    SELECT provenance_bucket, count(*) AS rows
    FROM governed WHERE subject_id = '{SUBJECT}' GROUP BY 1
""")).to_pylist()
print(f"Erasure request from {SUBJECT}: {before} rows, used as: {usage}")

dt = DeltaTable(GOVERNED)
dt.delete(f"subject_id = '{SUBJECT}'")
after_dt = DeltaTable(GOVERNED)
con.register("governed_after", after_dt.to_pyarrow_table())
after = con.sql(f"SELECT count(*) FROM governed_after WHERE subject_id = '{SUBJECT}'").fetchone()[0]

print(f"\nRows for {SUBJECT}: {before} → {after}")
print(f"Table version: {corpus_version} → {after_dt.version()}")
print(f"""
Note the tension the slide flags: time travel means v{corpus_version} STILL contains
the deleted rows. Removing rows from the current version does not remove old
physical files. Retention and VACUUM must be considered separately (NB6),
as must any copies or derived artifacts outside this table.""")

Erasure request from user_007: 8 rows, used as: [{'provenance_bucket': 'licensed', 'rows': 1}, {'provenance_bucket': 'UNCLASSIFIED', 'rows': 5}, {'provenance_bucket': 'scraped_optout_checked', 'rows': 1}, {'provenance_bucket': 'synthetic', 'rows': 1}]



Rows for user_007: 8 → 0
Table version: 0 → 1

Note the tension the slide flags: time travel means v0 STILL contains
the deleted rows. Removing rows from the current version does not remove old
physical files. Retention and VACUUM must be considered separately (NB6),
as must any copies or derived artifacts outside this table.


In [16]:
# Xoá ở version hiện tại có xoá thật không? So với run đã pin ở v{corpus_version}
gv = DeltaTable(GOVERNED)
for h in gv.history():
    m = h.get("operationMetrics", {})
    print(f"  v{h['version']} {h['operation']:<7} deleted_rows={m.get('num_deleted_rows', '-')}  "
          f"files +{m.get('num_added_files')} -{m.get('num_removed_files')}")
con.register("governed_v0", DeltaTable(GOVERNED, version=corpus_version).to_pyarrow_table())
v0_rows = con.sql(f"SELECT count(*) FROM governed_v0 WHERE subject_id = '{SUBJECT}'").fetchone()[0]
v0_train = con.sql(f"""SELECT count(*) FROM governed_v0
                       WHERE subject_id = '{SUBJECT}' AND provenance_bucket <> 'UNCLASSIFIED'""").fetchone()[0]
print(f"\n{SUBJECT} ở version hiện tại      : {after}")
print(f"{SUBJECT} qua time travel v{corpus_version}  : {v0_rows}   ← vẫn đọc được")
print(f"  trong đó nằm trong tập trainable mà model_card (corpus_version={corpus_version}) đã dùng: {v0_train}")
n_disk = len(list(Path(GOVERNED).rglob("*.parquet")))
print(f"parquet files on disk: {n_disk}  |  referenced now: {len(gv.file_uris())}")

  v1 DELETE  deleted_rows=8  files +4 -4
  v0 WRITE   deleted_rows=-  files +5 -0



user_007 ở version hiện tại      : 0
user_007 qua time travel v0  : 8   ← vẫn đọc được
  trong đó nằm trong tập trainable mà model_card (corpus_version=0) đã dùng: 3
parquet files on disk: 9  |  referenced now: 5


### 📝 Nhận xét — Xoá subject: version hiện tại ≠ mọi version

- `user_007` yêu cầu xoá: **8 dòng**; provenance cho biết ngay chúng thuộc bucket nào — 1 `licensed`, 1 `synthetic`, 1 `scraped_optout_checked`, **5 `UNCLASSIFIED`**. Sau `DELETE`: **8 → 0** ở version hiện tại, version 0 → 1 ✅.
- `DELETE` chỉ ghi lại **4/5 file** (`files +4 -4`) — đúng 4 partition chứa dữ liệu của `user_007`; partition `public_domain` không bị đụng ⇒ partition theo provenance giới hạn phạm vi copy-on-write.
- **Nhưng time travel về v0 vẫn đọc được đủ 8 dòng** (9 file trên đĩa, chỉ 5 được tham chiếu) — giống NB7. Muốn xoá thật phải **VACUUM** sau retention (NB6).
- **Mâu thuẫn quan trọng:** model card đã **pin `corpus_version = 0`** — chính version còn chứa dữ liệu của `user_007`, trong đó **3 dòng thuộc tập trainable** đã được dùng để train. Như vậy:
  - Giữ v0 để **tái lập** run ⇔ vẫn lưu dữ liệu người dùng đã yêu cầu xoá.
  - VACUUM để **xoá thật** ⇒ replay v0 không còn khả thi.
  - Và model đã train trên 3 dòng đó **không tự quên** — có thể cần train lại hoặc ghi nhận ngoại lệ.
  Reproducibility và quyền được xoá **xung đột trực tiếp**; cần chính sách retention, ghi nhận yêu cầu xoá trong model card, và xử lý cả các bản sao/artefact phái sinh (index, backup, model). Test ở đây chỉ chứng minh xoá khỏi version hiện tại — không chứng minh tuân thủ pháp lý.


## ✅ NB8 pass criteria

| Check | Target |
|---|---|
| Trajectory medallion | Silver partitioned by `agent_version`; Gold has both policies |
| Version pin | Replay at the pinned version matches the recorded step count |
| Cached list_tables | 5 agent turns → 1 catalog round-trip |
| MCP human-in-the-loop | Destructive call returns `input_required` before approval |
| MCP tasks | `submit_scan` → poll → `completed` |
| Provenance | all 4 illustrative lab buckets exist as partitions; UNCLASSIFIED excluded |
| Erasure | Subject rows = 0 in the current version, and you can say which bucket they were in |

In [17]:
checks = {
    "silver partitioned by agent_version": len(list(Path(SILVER).glob("agent_version=*"))) == 2,
    "gold covers both policies":           gold.num_rows == 2,
    "pinned version step count matches":   pinned.count() == training_run["n_steps_seen"],
    "5 turns → 1 catalog read":            mcp.catalog_reads == 1,
    "destructive needs confirmation":      attempt["resultType"] == "input_required",
    "confirmed call proceeds":             approved["resultType"] == "ok",
    "tasks poll completes":                st["status"] == "completed",
    "all 4 lab buckets present":           len([x for x in parts if "UNCLASSIFIED" not in x]) == 4,
    "unclassified rows found":             unclassified > 0,
    "erasure removed subject rows":        after == 0 and before > 0,
}
for k, v in checks.items():
    print(f"  [{'PASS' if v else 'FAIL'}] {k}")
assert all(checks.values()), "NB8 incomplete — see FAIL rows above"
print("\nNB8 complete.")

  [PASS] silver partitioned by agent_version
  [PASS] gold covers both policies
  [PASS] pinned version step count matches
  [PASS] 5 turns → 1 catalog read
  [PASS] destructive needs confirmation
  [PASS] confirmed call proceeds
  [PASS] tasks poll completes
  [PASS] all 4 lab buckets present
  [PASS] unclassified rows found
  [PASS] erasure removed subject rows

NB8 complete.


### 🎯 Kết luận NB8

Lakehouse là **system-of-record cho agent**: trajectory đi qua Bronze → Silver (partition theo `agent_version`) → Gold (so sánh 2 policy), và một số version được **pin** vào training run cho phép replay đúng 1.578 bước — bổ sung fingerprint cho thấy nội dung cũng khớp, và phát hiện 400 dòng "mới" thực ra là bản sao. Lớp MCP mô phỏng minh hoạ hình dạng ranh giới agent ↔ dữ liệu: cache `list_tables` (5 lượt → 1 lần đọc catalog), `input_required` trước thao tác phá huỷ (nhưng cờ xác nhận do bên gọi tự truyền — không phải phân quyền), task poll cho job dài và metering theo tool. Provenance được ghi thành cột + partition (4 bucket minh hoạ, 334 UNCLASSIFIED bị loại khỏi tập huấn luyện), với giới hạn rõ ràng của mapping. Cuối cùng, xoá subject cho thấy xung đột cốt lõi: **version được pin để tái lập chính là version còn giữ dữ liệu phải xoá** — reproducibility, retention và quyền được xoá phải được thiết kế cùng nhau.
